In [ ]:
%load_ext autoreload
%autoreload 2
import prismo as pm

from prismo.protocols import (
    purge_common_inlet,
    pattern_anti_gfp,
    purge_block_inlets,
    deadend_fill,
)

from prismo.utils import (
    sleep
)

import logging
from collections.abc import Iterator

from prismo.devices.microfluidic import Chip
from prismo.run import run_async
from prismo.utils import sleep

from tqdm import tqdm
import datetime
import matplotlib.pyplot as plt
import napari  
import numpy as np
import threading
import time

Create config

In [ ]:
config = {
    # The zyla camera connected to the microscope.
    # More details: https://github.com/Karl-Krauth/prismo/blob/main/src/prismo/devices/zyla.py#L4
    "camera": {
        "device": "zyla_camera",
        "flip": "lr",
    },
    # The Kinetix camera. Switch between cameras with use_camera("zyla") / use_camera("kinetix").
    "kinetix": {
        "device": "kinetix_camera",
        "flip": "both",
    },
    # The light engine to illuminate samples.
    # More details: https://github.com/Karl-Krauth/prismo/blob/main/src/prismo/devices/lumencor.py
    "light": {
        "device": "sola_light",
        "port": "COM5"
    },
    # The filter turret which decides which filter cube is currently active.
    # More details: https://github.com/Karl-Krauth/prismo/blob/main/src/prismo/devices/ti2.py#L1
    "filter": {
        "device": "ti2_filter1",
        # This names the filter cubes in the turret in order from 1-6.
        # The list always needs to assign names to all 6 cubes.
        # IMPORTANT NOTE: You probably need to modify this since the
        # filter cubes have probably moved since this demo notebook was written.
        "states": ["gfp", "cy5", "NADHTi",  "SOLA_DANAC", "empty5", "empty6"],
    },
    # The objective turret which decides the level of magnification.
    # More details: https://github.com/Karl-Krauth/prismo/blob/main/src/prismo/devices/ti2.py#L82
    "objective": {
        "device": "ti2_objective",
        "states": ["Apo4x", "10x", "empty1", "40x", "empty2", "Fluor4x"],
        "zooms": [4, 10, 0, 40, 0, 4]
    },
    # The device which allows you to modify what ist in focus.
    # More details: https://github.com/Karl-Krauth/prismo/blob/main/src/prismo/devices/ti2.py#L159
    "focus": {
        "device": "ti2_focus",
    },
    # The shutter which blocks light between the sample and the microscope when closed.
    # More details: https://github.com/Karl-Krauth/prismo/blob/main/src/prismo/devices/ti2.py#L128
    "shutter": {
        "device": "ti2_shutter1"
    },
    # The lightpath which selects which camera images of the sample go to.
    # More details: https://github.com/Karl-Krauth/prismo/blob/main/src/prismo/devices/ti2.py#L42
    "lightpath": {
        "device": "ti2_lightpath",
        # Ti2 ports in order: eye, right, aux, left.
        # Kinetix is on the right port, Zyla on the left.
        "states": ["eye", "kinetix", "aux", "zyla"],
    },
    # The stage controlling the sample position over the camera.
    # More details: https://github.com/Karl-Krauth/prismo/blob/main/src/prismo/devices/asi.py#L3
    "stage": {
        "device": "asi_stage",
        "port": "COM4"
    },
    "light_source": { 
        "device": "ti2_lightselector",
        "states": ["sola", "retra"],
    },
    # Valves refer to the pressure-driven manifold valves.
    # If you aren't running chips you can safely skip the rest of this cell.
    # More details: https://github.com/Karl-Krauth/prismo/blob/main/src/prismo/devices/microfluidic.py#L4
    "valves": { 
        "device": "microfluidic_valves",
         # You shouldn't ever need to change the IP address.
        "ip": "192.168.1.2",
    },
    
    "d1_ps_2block_v3": { 
        "device": "microfluidic_chip", # what should this be
        # A mapping from chip inlets to the control valve that's plugged into them.
        # IMPORTANT NOTE: You probably need to update this.
        # See the FAQ below for how to configure this mapping.
        "mapping": {
            "in0": 0,
            "in1": 1,
            "in2": 2,
            "in3": 3,
            "in4": 4,
            "in5": 5,
            "in6": 6,
            "waste7": 7,

            "inlet": 8,
            "neckL": 9,
            "butL": 10,
            "sandL": 11,
            "sandR": 12,
            "butR": 13,
            "neckR": 14,
            "out": 15,
        }
    },
    # # PS1.8k v2 device 2 (one neck)
    # "d2_ps_2block_v2": {
    #     "device": "microfluidic_chip",
    #     "mapping": {
    #         "in9": 16,
    #         "in8": 17,
    #         "in7": 18,
    #         "in6": 19,
    #         "in5": 20,
    #         "in4": 21,
    #         "in3": 22,
    #         "in2": 23,
    #         "waste": 24,

    #         "inlet": 25,
    #         "butL": 26,
    #         "sandL": 27,
    #         "sandR": 28,
    #         "butR": 29,
    #         "neck": 30,
    #         "out": 31,
    #     }
    # },
}

def use_camera(cam):
    c.camera = {"zyla": "camera", 
                "kinetix": "kinetix"}[cam]
    c.lightpath = cam
    c.wait()

Create controller object

In [ ]:
c = pm.load(config, path="C:/Program Files/Micro-Manager-2.0")
c.camera.binning = 2 # for 2x2 binning

# 16-bit imaging
c._core.setProperty("camera", "Sensitivity/DynamicRange", 
                    "16-bit (low noise & high well capacity)")

### Camera config

In [ ]:
c.camera.binning = 2 # for 2x2 binning

# zyla 16-bit imaging
c._core.setProperty("camera", "Sensitivity/DynamicRange", 
                    "16-bit (low noise & high well capacity)")

# Kinetix: 16-bit "Dynamic Range" port, and turn off the despeckle filters
# so pixel values aren't altered.
ports = c._core.getAllowedPropertyValues("kinetix", "Port")
c._core.setProperty("kinetix", "Port", next(p for p in ports if "Dynamic" in p))

for prop in c._core.getDevicePropertyNames("kinetix"):
    # if prop.startswith("PP") and prop.endswith("ENABLED"):
    if "DESPECKLE" in prop and prop.endswith("ENABLED"):
        c._core.setProperty("kinetix", prop, "No")
for prop in ["Port", "ReadoutRate", "BitDepth"]:
    print(f"kinetix {prop}: {c._core.getProperty('kinetix', prop)}")

###### Fixes for banding from Gemini #######
# 3. Ensure the hardware baseline factory gain mapping is strictly locked
c._core.setProperty("kinetix", "Gain", "1-Standard")

# FIX 1: Use fast memory Callbacks instead of slow CPU Polling
c._core.setProperty("kinetix", "AcquisitionMethod", "Callbacks")

# FIX 2: Enforce true internal hardware clocking
c._core.setProperty("kinetix", "TriggerMode", "Internal Trigger")

# FIX 3: Force the exposure output signaling to wait for ALL rows 
c._core.setProperty("kinetix", "ExposeOutMode", "All Rows")

# FIX 4: Safety reset of any accidental line delays leftover in memory
c._core.setProperty("kinetix", "ScanMode", "Auto")
##################################

assert c._core.getProperty("kinetix", "BitDepth") == "16"

# Function to change camera
def use_camera(cam):
    c.camera = {"zyla": "camera", 
                "kinetix": "kinetix"}[cam]
    c.lightpath = cam
    c.wait()

Create filter cube params dict

In [ ]:
filter_cube_params = {
    "gfp": {
        "cube_idx": 36,
        "ex": "488/40",
        "em": "525/50",
        "di": "495",
    },
    "cy5": {
        "cube_idx": 23,
        "ex": "618/50",
        "em": "698/70",
        "di": "652",
    },
    "NADHTi": {
        "cube_idx": 15,
        "ex": "377/50",
        "em": "470/100",
        "di": "409",
    },
}

### Start live imaging session

In [ ]:
use_camera('kinetix')
g = pm.live(c)

In [ ]:
c.exposure = 500

### Create chip object

In [ ]:
chip_d1 = c.d1_ps_2block_v3

### Set chip position

In [ ]:
d1_top_left = [-6099.5,  4497.5]
d1_bottom_right = [10276.6, 21788.5]

In [ ]:
c.xy

In [ ]:
g.quit()

### Setup
After connecting lines, iterate opening and closing all valves

In [ ]:
# actuate inlet valves
for i in range(20):
    print(f"i = {i}")
    for v in ["in1", "in2", "in3", "in4", "in5", "in6", "waste7"]:
        chip_d1[v] = "open"
    time.sleep(0.5)
    for v in ["in1", "in2", "in3", "in4", "in5", "in6", "waste7"]:
        chip_d1[v] = "closed"
    time.sleep(0.5)

In [ ]:
# blocks and inlets
for i in range(20):
    print(f"i = {i}")
    for v in ["inlet", "neckL", "butL", "sandL", "sandR", "butR", "neckR", "out"]:
        chip_d1[v] = "open"
    time.sleep(2)
    for v in ["inlet", "neckL", "butL", "sandL", "sandR", "butR", "neckR", "out"]:
        chip_d1[v] = "closed"
    time.sleep(2)

### Patterning

In [ ]:
from prismo.protocols.patterning import pattern_anti_egfp_high_low, protocol_running

if "p" in globals() and protocol_running(p):
    raise RuntimeError("A protocol is still running in p; wait for it (p.join()) first.")
    
p = pattern_anti_egfp_high_low(
    chip_d1,
    waste="waste7",
    bbsa="in6",
    na="in5",
    anti_egfp_high="in4",
    anti_egfp_low="in3",
    pbs="in1",
    high_side="L",
    outlet="out",
    info={
        "anti_egfp_high": "100 ug/mL bVHH", 
        "anti_egfp_low": "16.7 ug/mL bVHH, 1.5 mg/mL bBSA"
    },
)

In [ ]:
p.quit()

### Save device corners

In [ ]:
top_left_d1_zyla = (-6980.20, 3559.30)
bottom_right_d1_zyla = (11811.60, 23004.80)

### Background images

GFP background

In [ ]:
from tqdm.auto import tqdm

def snap(channel="gfp", exposure=500, power=100):
    c.filter = channel
    c.light_source = 'sola'
    c.light = power

    c.exposure = exposure
    c.shutter = "open"
    img = c.snap()
    c.shutter = "closed"
    return img

exposures = [500]

def acq_func(g, xs, ys):
    # Row and col are special variables which tells prismo to stitch images together in the GUI.
    imgs = g.array("tiled_images", exposure=exposures, row=ys, col=xs)
    imgs.coords["seconds"] = (("exposure", "row", "col"), np.zeros((len(exposures), len(ys), len(xs))))
    start_time = time.time()

    pbar = tqdm(total=len(ys) * len(xs), desc="Tiles", unit="tile")
    for i, y in enumerate(ys):
        for j, x in enumerate(xs):
            pbar.set_postfix(row=f"{i + 1}/{len(ys)}", col=f"{j + 1}/{len(xs)}")
            # Move the stage to the new position.
            c.xy = (x, y)
            c.wait()
            for e in exposures:
                imgs['seconds'].loc[e, y, x] = time.time() - start_time
                imgs.loc[e, y, x] = snap(exposure=e)
            pbar.update(1)
            yield
        # Flip the xs array so we image in a snaking pattern.
        xs = np.flip(xs)
    pbar.close()

In [ ]:
use_camera("kinetix")
g = pm.tiled_acq(c, "GFP/gfp_bg_kinetix_500ms", acq_func, overlap=0.1,
                 top_left=d1_top_left, 
                 bot_right=d1_bottom_right
                )

In [ ]:
g.quit()

In [ ]:
use_camera("kinetix")
g = pm.live(c)

### Post expression GFP image

In [ ]:
from prismo.protocols.imaging import tile_image

g = tile_image(c, "GFP/egfp_post_30min_expression_kinetix_500ms",
               channel="gfp", 
               top_left=d1_top_left, 
               bot_right=d1_bottom_right
              )

### Binding

In [ ]:
from prismo.protocols.binding import track_binding

try:
    g.quit()  # close leftover live view / tilings
except NameError:
    pass

# Set the valves yourself first (buttons open, sandwiches closed).
g = track_binding(
    c, "GFP/egfp_binding_kinetix_500ms",
    top_left=d1_top_left, 
    bot_right=d1_bottom_right,
    passes=30,
)

In [ ]:
g.quit()

In [ ]:
g = pm.live(c)

### SDS wash

In [ ]:
from prismo.protocols.washes import sds_wash_series

sds_wash_series(c, chip_d1, 
                sds="in3", 
                pbs="in1", 
                waste="waste7", 
                top_left=d1_top_left, 
                bot_right=d1_bottom_right
               )

### Final clean post SDS GFP image

In [ ]:
from prismo.protocols.imaging import tile_image

g = tile_image(c, "GFP/egfp_post_SDS_clean_kinetix_500ms",
               channel="gfp", 
               top_left=d1_top_left, 
               bot_right=d1_bottom_right
              )

### Cy5 background

In [ ]:
from prismo.protocols.imaging import tile_image

g = tile_image(c, "Cy5/cy5_bg_kinetix_500ms",
               channel="cy5", 
               top_left=d1_top_left, 
               bot_right=d1_bottom_right
              )

### SNAP labeling kinetics
Flow on Cy5-BG before running cell

In [ ]:
from prismo.protocols.assays import snap_labeling_kinetics

# Close leftover sessions so they don't slow the acquisition.
try:
    g.quit()
except NameError:
    pass

g = snap_labeling_kinetics(
    c, chip_d1, "Cy5/cy5_SNAP_kinetics_kinetix_500ms",
    incubations_s=[1, 5, 5, 10, 10, 10, 120, 120, 120, 300, 300, 300, 300],   # buttons-open time before each image
    top_left=d1_top_left, 
    bot_right=d1_bottom_right,
    info={"ligand": "Cy5-BG", "conc_uM": 1},
)

In [ ]:
c.shutter = "open"

In [ ]:
c.shutter = "closed"

In [ ]:
g = pm.live(c)

### Post-labeling Cy5 image

In [ ]:
from prismo.protocols.imaging import tile_image

g = tile_image(c, "Cy5/cy5_post_labeling_kinetix_500ms",
               channel="cy5", 
               top_left=d1_top_left, 
               bot_right=d1_bottom_right
              )

In [ ]:
g.quit()

### Single NADH shutter test

In [ ]:
from prismo.protocols.imaging import tile_image

g = tile_image(c, "kinetix_nadh_test",
               channel="NADHTi", 
               top_left=d1_top_left, 
               bot_right=d1_bottom_right
              )

In [ ]:
g.quit()

### NADH Kinetics

In [ ]:
# %% Snap + acquisition function
import datetime
import time

import numpy as np
from tqdm.auto import tqdm


def snap(channel="NADHTi", exposure=500, power=100):
    c.filter = channel
    c.light_source = 'sola'
    c.light = power

    c.exposure = exposure
    c.shutter = "open"
    img = c.snap()
    c.shutter = "closed"
    return img


def camera_properties(cam):
    """Every Micro-Manager property of the active camera (port, readout rate, gain, temperature...)."""
    props = {}
    for p in c._core.getDevicePropertyNames(cam):
        try:
            props[p] = c._core.getProperty(cam, p)
        except Exception as e:
            props[p] = f"<unreadable: {e}>"
    return props


def make_kinetics_acq_func(substrate, substrate_conc, n_acquisitions=15,
                           channel="NADHTi", exposure=500, power=100):
    def acq_func(g, xs, ys):
        # Set the light path up first so the attrs below are read back from the devices.
        c.filter = channel
        c.light_source = 'sola'
        c.light = power
        c.exposure = exposure
        c.wait()

        # Row and col are special variables which tells prismo to stitch images together in the GUI.
        imgs = g.array("tiled_images", time=n_acquisitions, row=ys, col=xs)

        # Per-tile elapsed seconds and wall-clock time (tiles in one pass are taken at different times).
        shape = (n_acquisitions, len(ys), len(xs))
        imgs.coords["seconds"] = (("time", "row", "col"), np.zeros(shape))
        imgs.coords["timestamp"] = (("time", "row", "col"), np.full(shape, np.datetime64("NaT"), dtype="datetime64[ns]"))
        imgs.coords["timestamp"].encoding["units"] = "microseconds since 1970-01-01"
        imgs.coords["timestamp"].encoding["dtype"] = "int64"

        # Experiment.
        imgs.attrs["substrate"] = substrate
        imgs.attrs["substrate_conc"] = substrate_conc
        imgs.attrs["n_acquisitions"] = n_acquisitions
        imgs.attrs["info"] = f"{substrate_conc} {substrate}, {exposure}ms {channel}, {n_acquisitions} back-to-back passes"

        # Hardware identities, read back from the devices.
        cam = c.camera.name
        if c.filter not in filter_cube_params:
            print(f"Note: '{c.filter}' is not in filter_cube_params; filter_cube is saved as None.")
        imgs.attrs["objective"] = c.objective
        imgs.attrs["px_len_um"] = float(c.px_len)
        imgs.attrs["filter"] = c.filter
        imgs.attrs["filter_cube"] = filter_cube_params.get(c.filter)
        imgs.attrs["light"] = {"source": c.light_source, "device": "sola_light", "power": c.light}
        imgs.attrs["lightpath"] = c.lightpath
        imgs.attrs["camera"] = {
            "name": {"camera": "zyla"}.get(cam, cam),  # the Zyla's device name is 'camera'
            "binning": c.camera.binning,
            "exposure_ms": c.exposure,
            "properties": camera_properties(cam),
        }
        imgs.attrs["z"] = float(c.z)

        # Close sandwiches and open buttons (necks are left as they are).
        chip = chip_d1
        chip.sandL = 'closed'
        chip.sandR = 'closed'
        print("Closed sandwiches. Waiting 2 seconds...")
        time.sleep(2)

        chip.butL = 'open'
        chip.butR = 'open'
        print("Opened buttons. Beginning acquisition...")

        # Get the start time of the experiment
        start_time = time.time()
        imgs.attrs["start_time"] = datetime.datetime.now().isoformat()

        pbar = tqdm(total=n_acquisitions * len(ys) * len(xs), desc="Tiles", unit="tile")
        xs0 = np.array(xs)
        for t in range(n_acquisitions):
            # Start every pass at the top left so each tile is sampled once per pass at a constant interval
            # (otherwise an odd row count leaves xs flipped and the next pass starts top right).
            xs = xs0
            curr_time = time.time() - start_time
            print(f"Pass {t}: {curr_time:.1f}s ({curr_time / 60:.2f}m)")

            for i, y in enumerate(ys):
                for j, x in enumerate(xs):
                    pbar.set_postfix(t=f"{t + 1}/{n_acquisitions}", row=f"{i + 1}/{len(ys)}", col=f"{j + 1}/{len(xs)}")

                    # Move the stage to the new position.
                    c.xy = (x, y)
                    c.wait()

                    imgs['seconds'].loc[t, y, x] = time.time() - start_time
                    imgs['timestamp'].loc[t, y, x] = np.datetime64(datetime.datetime.now())
                    imgs.loc[t, y, x] = snap(channel=channel, exposure=exposure, power=power)
                    pbar.update(1)
                    yield
                # Flip the xs array so we image in a snaking pattern.
                xs = np.flip(xs)

        pbar.close()
        print('Done with acquisition.')

    return acq_func

In [ ]:
# %% Run (set the substrate concentration each time)
substrate = "BzP"
substrate_conc = "78.125uM"
n_acquisitions = 20      # back-to-back tilings, no wait in between
channel = "NADHTi"       # filter cube
exposure = 500          # ms; keep short: this wavelength damages proteins
power = 100              # SOLA white level (0-100)
camera = "kinetix"

# Close leftover sessions so they don't slow the acquisition.
try:
    g.quit()
except NameError:
    pass

use_camera(camera)
conc_tag = substrate_conc.replace(".", "_")
g = pm.tiled_acq(c, f"Assay/NADH_kinetics_{conc_tag}_{substrate}_{camera}_{exposure}ms",
                 make_kinetics_acq_func(substrate, substrate_conc, n_acquisitions,
                                        channel=channel, exposure=exposure, power=power),
                 overlap=0.1, 
                 top_left=d1_top_left, 
                 bot_right=d1_bottom_right
                )


In [ ]:
g = pm.live(c)

For final condition, run this one too

In [ ]:
# Wait for the running imaging session (g) to finish, then flush with K-only buffer.
print("Waiting for the imaging run to finish...")
g.join()
print("Imaging finished.")

chip = chip_d1
buffer = "in1"        # K-only buffer (MOPS + KOAc)
flush_s = 5 * 60

# Close all valves except the necks.
for v in chip.valves:
    if v.split("[")[0] not in ("neckL", "neckR"):
        chip[v] = "closed"
print("Closed all valves (necks untouched):", {n: chip[n] for n in ("neckL", "neckR")})

# Purge air from the buffer line to waste.
chip[buffer] = "open"; chip.waste7 = "open"
time.sleep(5)
chip.waste7 = "closed"
print(f"Purged {buffer} to waste7.")

# Flow buffer through the device: sandwiches open, buttons closed.
chip.sandL = "open"; chip.sandR = "open"
chip.inlet = "open"; chip.out = "open"
print(f"Flushing K-only buffer ({buffer}) through the device for {flush_s // 60} min...")
for _ in tqdm(range(flush_s), desc="Buffer flush", unit="s"):
    time.sleep(1)

# End dead-ended in buffer.
chip.out = "closed"
print("Done: dead-ended in K-only buffer (outlet closed, buttons closed).")

### Standards

In [ ]:
from prismo.protocols.imaging import tile_image

conc = "0uM"
exposure = 500

# Close leftover sessions so they don't slow the acquisition.
try:
    g.quit()
except NameError:
    pass

# Close sandwiches, then open buttons (necks are left as they are).
chip = chip_d1
chip.sandL = "closed"
chip.sandR = "closed"
print("Closed sandwiches. Waiting 2 seconds...")
time.sleep(2)

chip.butL = "open"
chip.butR = "open"
print("Opened buttons. Starting tiling...")

g = tile_image(c, f"Standards/NADH_standard_{conc}_{exposure}ms",
               channel="NADHTi", 
               exposure=exposure,
               top_left=d1_top_left, 
               bot_right=d1_bottom_right
              )

In [ ]:
g.quit()

For final condition, run this one too

In [ ]:
# Wait for the running imaging session (g) to finish, then flush with K-only buffer.
print("Waiting for the imaging run to finish...")
g.join()
print("Imaging finished.")

chip = chip_d1
buffer = "in1"        # K-only buffer (MOPS + KOAc)
flush_s = 5 * 60

# Close all valves except the necks.
for v in chip.valves:
    if v.split("[")[0] not in ("neckL", "neckR"):
        chip[v] = "closed"
print("Closed all valves (necks untouched):", {n: chip[n] for n in ("neckL", "neckR")})

# Purge air from the buffer line to waste.
chip[buffer] = "open"; chip.waste7 = "open"
time.sleep(5)
chip.waste7 = "closed"
print(f"Purged {buffer} to waste7.")

# Flow buffer through the device: sandwiches open, buttons closed.
chip.sandL = "open"; chip.sandR = "open"
chip.inlet = "open"; chip.out = "open"
print(f"Flushing K-only buffer ({buffer}) through the device for {flush_s // 60} min...")
for _ in tqdm(range(flush_s), desc="Buffer flush", unit="s"):
    time.sleep(1)

# End dead-ended in buffer.
chip.out = "closed"
print("Done: dead-ended in K-only buffer (outlet closed, buttons closed).")